In [191]:
import requests
import pandas as pd

url = "http://localhost:8000/upload_config_from_csv/1/"

input_csv = "alzheimers_phi.csv"

files = {"file": ("all_tables.csv", open(input_csv, "rb"), "text/csv")}

response = requests.post(url, files=files)

print(response.status_code, response.json())

200 {'success': True, 'message': 'Configuration uploaded successfully.'}


In [192]:
df = pd.read_csv(input_csv)
tables = df['TABLE_NAME'].unique()
len(tables)

50

In [193]:
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

36

In [194]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

14

In [195]:
static_df = no_notes_df[no_notes_df['DE_IDENTIFICATION_RULE'] == 'STATIC_OFFSET']
static_tables = list(set(static_df['TABLE_NAME'].to_list()))
len(static_tables)

3

In [196]:
import os
import django
import sys
# Set up Django environment
sys.path.append('F:\Soorykant\deidentification\deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [197]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "PATIENT_ID"
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [198]:
to = TableDetailsModel.objects.get(table_name="edi_inv_cpt")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvoiceId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'edi_inv_cpt',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [199]:
to = TableDetailsModel.objects.get(table_name="edi_inv_diagnosis")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvoiceId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'edi_inv_diagnosis',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [200]:
to = TableDetailsModel.objects.get(table_name="hcfa")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'hcfa',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [201]:
to = TableDetailsModel.objects.get(table_name="hl7labdatadetail")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
   'source_column': 'ReportId',
   'reference_table': 'labdata'}],
 'source_table': 'hl7labdatadetail',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.save()

In [202]:
to = TableDetailsModel.objects.get(table_name="hl7labnotes")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
   'source_column': 'reportid',
   'reference_table': 'labdata'}],
 'source_table': 'hl7labnotes',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.save()

In [203]:
to = TableDetailsModel.objects.get(table_name="hcfa")
to.table_details_for_ui['reference_mapping']

{'conditions': [{'column_name': 'Id',
   'source_column': 'InvId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'hcfa',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}

In [204]:
TableDetailsModel.objects.filter(table_status=0).count()

3

In [205]:
TableDetailsModel.objects.filter(table_status=1).count()

0

In [206]:
TableDetailsModel.objects.filter(table_status=2).count()

48

In [207]:
TableDetailsModel.objects.filter(table_status=3).count()

0

In [208]:
TableDetailsModel.objects.filter(table_status=3)

<QuerySet []>

In [239]:
Task.objects.filter(status=0).count()

0

In [240]:
Task.objects.filter(status=1).count()

0

In [235]:
Task.objects.filter(status=2).count()

50

In [236]:
Task.objects.filter(status=-1).count()

0

In [238]:
# Task.objects.filter(status=-1, arguments__table_id=9).count()

In [219]:
# Task.objects.filter(arguments__table_id=4)[0].remarks

In [220]:
# t = TableDetailsModel.objects.get(table_name='hcfa')
# t.id

In [221]:
# Chain.objects.all().delete()

In [222]:
from django.conf import settings
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION, settings.DEFAULT_OFFSET_VALUE

(1000, 34)

In [223]:
tables = ['hcpcscode_base', 'labloinccodes']
completed = ['allergies', 'annualnotes', 'billingdata', 'edi_invoice', 'enc', 'encaddendums', 
             'encounterdata', 'encounters', 'family', 'hpi', 'immunizations', 'inpatientvisit', 
             'interactionnotes', 'labdata', 'notes', 'oldrxmain', 'patients', 'procedurespl', 'ptinstruction', 'review', 'social', 
             'structhpi', 'structsocialhistory', 'surgicalhistory', 'telenc', 'treatmentnotes', 'users', 'vitalshistory',
            'assessment_notes_history', 'doctors', 'edi_dfr_info', 'icd_9', 'icd10cm_desc', 'items', 'lablist', 
             'oldrxdetail', 'progressnotes_decryptfinal', 'properties', 'structured_data', 'structccmr', 'cpt_validcodes',
            ]
relation_tables = ['edi_inv_cpt', 'edi_inv_diagnosis', 'hl7labdatadetail', 'hl7labnotes', 'hcfa', 'ndclookupenteries', 'rx_medication_alert']
len(tables), len(completed), len(relation_tables)

(2, 41, 7)

In [224]:
totalrun = 0
for table in tables:
    tableobj = TableDetailsModel.objects.get(table_name=table)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    # if tableobj.table_status in [2]:
    #     continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()
    tasks, chain = create_deidentification_task(tableobj, False)
    totalrun += 1
    print(tableobj.rows_count)

[]
[]


2025-06-01 18:50:01,917 - deIdentification.nd_logger - INFO - inside marking as in progress if required


10089
[]
[]


2025-06-01 18:50:02,006 - deIdentification.nd_logger - INFO - inside marking as in progress if required


32460
